In [43]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

'''-----------------------------------1. Download the sentiment reviews dataset.-----------------------------------'''
#df = pd.read_csv('sentiment_reviews.csv')
df = pd.read_csv('test.csv', encoding='latin-1')
#print(df)

df = df.dropna(subset=['text'])

#print(review)
#print(label)

'''-----------------------------------2. Preprocess text using NLTK or regex (remove HTML tags, punctuation, stopwords).-----------------------------------'''
stop_words = set(stopwords.words('english'))
#print(stop_words)
lemmatize = WordNetLemmatizer()
def clean_text(text):
    text = text.lower()
    #print(text)
    text = re.sub(r'[^a-z\s]','', text)
    #print(text)

    text = text.split()
    #print(text)

    clean_text = []

    for word in text:
        if word not in stop_words:
            clean_text.append(lemmatize.lemmatize(word))

    return ' '.join(clean_text)

# Apply cleaning to the WHOLE dataset before splitting
df['cleaned_text'] = df['text'].apply(clean_text)

'''-----------------------------------3. Convert text to numerical vectors using TfidfVectorizer(max_features=5000).-----------------------------------'''
train_x, test_x, train_y , test_y = train_test_split(df['cleaned_text'], df['sentiment'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
vector_x = vectorizer.fit_transform(train_x)
vector_x_test = vectorizer.transform(test_x)

'''-----------------------------------4. Train MultinomialNB and LogisticRegression.-----------------------------------'''

#Naive Bayes Model
nb_model = MultinomialNB()
nb_model.fit(vector_x, train_y)
nb_predicts = nb_model.predict(vector_x_test)

nb_acc = accuracy_score(test_y, nb_predicts)
print(f"Naive Bayes Model accuracy: {nb_acc:.2f}")

#Logistic Regression Model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(vector_x, train_y)
lr_preds = lr_model.predict(vector_x_test)

lr_acc = accuracy_score(test_y, lr_preds)
print(f"Logistic Regression Model accuracy: {lr_acc:.2f}")
'''-----------------------------------5. Create a predict_sentiment() function that tests live custom user input phrases.-----------------------------------'''
def predict_sentiment(txt):
  cleaned = clean_text(txt)
  vectorized = vectorizer.transform([cleaned])
  nb_prediction = nb_model.predict(vectorized)[0]
  lr_prediction = lr_model.predict(vectorized)[0]

  print(f"Logistic Regression predicted: {lr_prediction}")
  print(f"Naive Bayes predicted: {nb_prediction}")


#clean_text(review[0])
feedback = input("Enter your Review:")
predict_sentiment(feedback)

'''
Final Performance Summary: NLP Sentiment Classification

i. Model Overview
We built an end-to-end Natural Language Processing (NLP) pipeline to classify customer reviews.
a.   Preprocessing: Text was lowercased, cleaned of punctuation using Regular Expressions, stripped of NLTK stopwords,
     and lemmatized using WordNetLemmatizer.
b.   Vectorization: Text was converted into numerical features using TF-IDF (max_features=5000).
c.   Models: Multinomial Naive Bayes and Logistic Regression.

ii. Evaluation Metrics
Due to the small size of the starter dataset, the accuracy scores (~60% for NB and ~64% for LR) are constrained by the
limited number of test samples. However, the models demonstrate clear learning.

iii. Interactive Helper Test (Live Custom Inputs)
The required `predict_sentiment(text)` function is fully functional. We tested it on custom, unseen inputs:
a.   Input: "the service was good, I enjoyed my vacations alot in murree" -> Predicted: Positive
b.   Input: "i hate it, the food was worse" -> Predicted: Negative

iv. Limitations & Future Improvements
During live testing, a complex sentence ("the food was good, but the server was very rude") was misclassified as Positive.
This highlights a known limitation of TF-IDF and Naive Bayes:
a.  Bag of Words: TF-IDF ignores word order, so it doesn't understand the contrast of the word "but".
b.  Out of Vocabulary (OOV): Because the training dataset was small, the word "rude" might have been unseen, so the model
    gave it a weight of zero. The highly weighted word "good" dominated the prediction.
c.  Production Fix: In a real-world system, using a larger dataset (e.g., 50,000 IMDb reviews) and adding N-grams (bigrams)
    would solve these issues.
'''

Naive Bayes Model accuracy: 0.60
Logistic Regression Model accuracy: 0.64
Enter your Review:i hate it, the food was worse
Logistic Regression predicted: negative
Naive Bayes predicted: negative
